In [112]:
# Programa: Simulacion de robots de limpieza en espacio M x M
# Autores:
# Juan Pablo Solís Gómez A01800430
# Israel González Huerta A01751433
# Fecha 2025-11-14

# Has multi-dimensional arrays and matrices.
# Has a large collection of mathematical functions to operate on these arrays.
import numpy as np

# Data manipulation and analysis.
import pandas as pd

# Data visualization tools.
import seaborn as sns

import mesa

import time

from mesa.discrete_space import CellAgent, OrthogonalMooreGrid

In [113]:
class DirtAgent(CellAgent):
    """Representa suciedad fija en una celda."""

    def __init__(self, model, cell):
        super().__init__(model)
        self.cell = cell      # celda donde está
        self.dirty = True     

    def step(self):
        # La suciedad no hace nada
        pass


In [114]:
class LimpiadorAgent(CellAgent):
    """robot que limpia suciedad o se mueve si la celda está limpia."""

    def __init__(self, model, cell):
        super().__init__(model)
        self.cell = cell
        self.moves = 0  # movimientos realizados por este robot

    def step(self):
        # 1. Buscar suciedad la celda actual
        dirt_here = [
            a for a in self.cell.agents
            if isinstance(a, DirtAgent) and a.dirty
        ]

        if dirt_here:
            # Limpiar = marcar los DirtAgent como no sucios
            for d in dirt_here:
                d.dirty = False
            # No se mueve en este paso
            return

        # 2. Si no hay suciedad, moverse a una celda vecina aleatoria (Moore)
        new_cell = self.cell.neighborhood.select_random_cell()

        # 3. Solo moverse si:
        if (
            new_cell is not None
            and new_cell is not self.cell
            and not any(isinstance(a, LimpiadorAgent) for a in new_cell.agents) #Checar si no hay otro agente
        ):
            self.cell = new_cell
            self.moves += 1
            self.model.total_moves += 1


In [115]:
class LimpiadorModel(mesa.Model):
    def __init__(self, width, height, n_agents, dirty_prob, seed=None):
        super().__init__(seed=seed)
        self.width = width
        self.height = height
        self.n_agents = n_agents
        self.dirty_prob = dirty_prob

        # Estadísticas
        self.step_count = 0
        self.total_moves = 0

        # Grid de vecindad Moore (8 vecinos)
        self.grid = OrthogonalMooreGrid(
            (width, height),
            torus=False,
            capacity=width * height * 3,
            random=self.random,
        )

        # Crear suciedad recorriendo toda la grid
        for cell in self.grid.all_cells:
            if self.random.random() < dirty_prob:
                dirt = DirtAgent(self, cell)
                self.agents.add(dirt)

        # Crear robots todos en (1,1)
        start_cell = self.grid[(1, 1)]
        for _ in range(n_agents):
            vac = LimpiadorAgent(self, start_cell)
            self.agents.add(vac)

    def remaining_dirt(self):
        """Número de celdas sucias (DirtAgents todavía marcados como dirty)."""
        return sum(
            1
            for a in self.agents
            if isinstance(a, DirtAgent) and a.dirty
        )

    def step(self):
        self.agents.shuffle_do("step")
        self.step_count += 1


In [116]:
M = 50               # ancho
N = 50                # alto
NUM_AGENTES = 100      # 
PORCENTAJE_SUCIA = 0.65   # % de celdas sucias
TIEMPO_MAX_SEG = 5 # máximo de segundos de ejecución


In [117]:
model = LimpiadorModel(
    width=M,
    height=N,
    n_agents=NUM_AGENTES,
    dirty_prob=PORCENTAJE_SUCIA
)

inicio = time.perf_counter()

while (
    model.remaining_dirt() > 0
    and (time.perf_counter() - inicio) < TIEMPO_MAX_SEG
):
    model.step()
fin = time.perf_counter()
TIEMPO_REAL = fin - inicio


In [118]:
total_cells = M * N
dirty_cells = model.remaining_dirt()
clean_cells = total_cells - dirty_cells

tiempo_hasta_limpiar = model.step_count
porcentaje_limpias = clean_cells / total_cells * 100
movimientos_totales = model.total_moves

print("Simulación terminada.")
print("Pasos ejecutados:", tiempo_hasta_limpiar)
print("Celdas sucias restantes:", dirty_cells)
print("Porcentaje de celdas limpias:", porcentaje_limpias)
print("Movimientos totales:", movimientos_totales)
print(f"Tiempo de ejecución: {TIEMPO_REAL: .4f}segundos")

Simulación terminada.
Pasos ejecutados: 1513
Celdas sucias restantes: 0
Porcentaje de celdas limpias: 100.0
Movimientos totales: 125956
Tiempo de ejecución:  1.2752segundos
